In [1]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

#Change the path of your loca repository
sys_path = "Desktop/Practicas_Movilidad"
sys.path.insert(0, str(Path.home() / sys_path))
from multimodalpy.get_network import main

from multimodalpy.network_qa import (
    run_all_checks_osm,
    summary_matrix,
    style_summary_matrix,
    show_issues,
)

In [2]:
AREA_NAME = "Madrid" #### CHANGE ###

CRS = "epsg:25830"
MODES = ["driving", "walking", "bike", "bus", "train"]
NAP_API_KEY= "44a2e8d8-fab3-4d87-940d-d7788a7dac94"

output_folder = Path.home() / "Desktop/descargas_multimodalpy/output_Madrid" #### CHANGE ###
boundaries = "C:/Users/Usuario/Desktop/Practicas_Movilidad/multimodalpy/data/boundaries/layer_{}/layer_{}.shp".format(AREA_NAME, AREA_NAME) #### CHANGE ###


### Checks:

**Nodes**
* **nodes: expected columns present:** the nodes table contains all the expected columns (node_id, node_role, geometry).

* **no fully duplicate rows:** no row of nodes is identical to another in all its columns.
* **node_id is unique:** no node_id is duplicated.
* **node_id not null:** no node has an empty or NaN node_id.
* **geometry not null:** no node has an empty geometry.
* **node_role in [...]:** the value of node_role is one of the four valid values (endpoint, intersection, linear_vertex, through_endpoint).
* **geometry parses as valid POINT:** the geometry can be parsed and is a valid POINT (not a line, not corrupted).

**Edges**
* **edges: expected columns present:** the edges table contains all the columns expected by the OSM schema.

* **edge_id not null:** no edge has an empty edge_id.
* **from_node_id / to_node_id not null:** no edge has an empty source/destination node.
* **from_node_id / to_node_id exists in nodes table:** every node referenced by an edge actually exists in the nodes table (referential integrity).
* **no self-loop edges (from == to):** no edge starts and ends at the same node.
* **length not null / is non-negative:** the length column is present and contains no negative values.
* **tts (travel time) is non-negative:** the travel time is not negative.
* **oneway / reversed is boolean dtype:** these columns are of type True/False, not text (“true”/“1”/etc.).
* **geometry parses as valid LINESTRING:** the geometry of each edge is a valid line.
* **edge geometry endpoints match node coords:** the endpoints of the line match (within a tolerance) the coordinates of its from/to nodes.
* **length column matches geometry length:** the numerical value of ‘length’ matches (±2%) the actual calculated length of the geometry.
* **maxspeed numeric (...):** the speed limit is a number (or multiple values separated by ;, as is typical in OSM).
* **lanes numeric & positive (...):** the number of lanes is a positive number.
* **highway tag(s) within the recognised OSM vocabulary:** the value of the ‘highway’ tag is one of the recognised OSM tags (residential, primary, etc.), not an unusual or unexpected one.



In [3]:
STEM_BY_MODE = {
    "driving": "osm_driving",
    "walking": "osm_walking",
    "bike": "osm_bike",
    "train": "gtfs_20260921_000008_renfe_avld",  #### CHANGE ###
    "bus": "gtfs_20260921_000034_bus_alsa", #### CHANGE ###
}


def load_network(mode: str, fmt: str, gpkg_path=None):

    stem = STEM_BY_MODE[mode]
    folder = Path(output_folder)

    if fmt == "geojson":
        nodes = gpd.read_file(folder / f"{stem}_nodes.geojson")
        edges = gpd.read_file(folder / f"{stem}_edges.geojson")
    elif fmt == "shapefile":
        nodes = gpd.read_file(folder / f"{stem}_nodes.shp")
        edges = gpd.read_file(folder / f"{stem}_edges.shp")
    elif fmt == "geopackage":
        if gpkg_path is None:
            raise ValueError("fmt='geopackage' requiere gpkg_path=...")
        gpkg_path = Path(gpkg_path)
        nodes = gpd.read_file(gpkg_path, layer=f"{stem}_nodes")
        edges = gpd.read_file(gpkg_path, layer=f"{stem}_edges")
    else:
        raise ValueError(f"Formato desconocido: {fmt!r}")

    return nodes, edges


def run_checks_for_all_modes(fmt: str, gpkg_path=None) -> dict:

    results_by_mode = {}
    for mode in MODES:
        nodes, edges = load_network(mode, fmt, gpkg_path=gpkg_path)
        results_by_mode[mode] = run_all_checks_osm(nodes, edges, verbose=False)
    return results_by_mode

def show_results(results_by_mode: dict):
    matrix = summary_matrix(results_by_mode)
    return style_summary_matrix(matrix)


## GeoJSON

In [4]:
# Una sola llamada escribe los tres formatos: la descarga y la normalizacion
# se hacen una unica vez en vez de repetirse por formato.
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type=["geojson", "shapefile", "geopackage"],
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)


ConnectTimeout: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<HTTPSConnection(host='overpass-api.de', port=443) at 0x21621f8fa90>, 'Connection to overpass-api.de timed out. (connect timeout=180)'))

In [5]:
results_geojson = run_checks_for_all_modes("geojson")
show_results(results_geojson)

DataSourceError: C:\Users\Usuario\Desktop\descargas_multimodalpy\output_Madrid\osm_driving_nodes.geojson: No such file or directory

#### Analyse each check with errors:

In [6]:
show_issues(results_geojson["driving"], "maxspeed numeric (incl. ';'-separated values) when present").head(2)

NameError: name 'results_geojson' is not defined

In [7]:
show_issues(results_geojson["bus"], "node_role in ['endpoint', 'intersection', 'linear_vertex', 'through_endpoint']").head(2)

NameError: name 'results_geojson' is not defined

## Shapefile

In [8]:
# Ya escrito por la llamada unica a main() de la seccion GeoJSON.


In [9]:
results_shapefile = run_checks_for_all_modes("shapefile")
show_results(results_shapefile)

DataSourceError: C:\Users\Usuario\Desktop\descargas_multimodalpy\output_Madrid\osm_driving_nodes.shp: No such file or directory

## GeoPackage

In [10]:
# Ya escrito por la llamada unica a main() de la seccion GeoJSON.


In [11]:
gpkg_path = output_folder / "{}.gpkg".format("madrid") #### CHANGE ###
results_geopackage = run_checks_for_all_modes("geopackage", gpkg_path=gpkg_path)
show_results(results_geopackage)

DataSourceError: C:\Users\Usuario\Desktop\descargas_multimodalpy\output_Madrid\madrid.gpkg: No such file or directory